# Synthetic Conversation Data Generator
### Input Guardrails for Youth Mental Health Chatbot

This notebook generates labeled synthetic chat transcripts for training a risk-classification model on a youth mental health chatbot.

**Pipeline overview:**
1. Define a risk taxonomy (23 categories × low/high signal)
2. Build a persona library — both hand-crafted and LLM-generated
3. Simulate multi-turn conversations using two competing LLM agents (user + chatbot)
4. Auto-evaluate each conversation with an LLM-as-a-judge
5. Export a labeled CSV dataset for downstream classifier training

**Output schema:** `Conversation_id | Turns | Text | Category | Risk | language | label`

---
> ⚠️ **Ethical note:** All data is fully synthetic. No real user data is used. Personas are fictional composites designed to represent diverse Canadian youth demographics.

## 0. Setup & Imports

In [ ]:
import random
import os
import json
import time
import pandas as pd
from typing import List

# LLM provider — replace with your own wrapper or OpenAI client
# from project.providers.openai_provider import OpenAIProvider
# from project.providers.base import LLMMessage

## 1. LLM API Wrapper

Single function to call the underlying LLM. Swap out the provider here if needed.
- `temp`: controls creativity — kept higher (~0.8) for the user actor, lower (~0.3) for the chatbot
- `m_tokens`: max response length

In [ ]:
DEFAULT_API_ENDPOINT = "https://gpt-oss-120b-3e1ds.inference.buzzperformancecloud.com"

api_key = os.getenv("BUZZ_GPT_OSS_AUTH_TOKEN")
if not api_key:
    raise ValueError(
        "API key must be set via the BUZZ_GPT_OSS_AUTH_TOKEN environment variable"
    )


def call_llm_api(messages, system_prompt, temp, m_tokens=600):
    """
    Calls the LLM with a given message history and system prompt.

    Args:
        messages (list): List of dicts with 'role' and 'content' keys.
        system_prompt (str): The system prompt to use for this call.
        temp (float): Sampling temperature. Higher = more creative.
        m_tokens (int): Max tokens in the response.

    Returns:
        str: The model's response text.
    """
    provider = OpenAIProvider(
        temperature=temp,
        max_tokens=m_tokens
    )
    provider.system_prompt = system_prompt

    formatted_messages: List[LLMMessage] = [
        LLMMessage(role=m["role"], content=m["content"])
        for m in messages
    ]
    response = provider.generate_sync(formatted_messages)
    return response.content.strip()

## 2. Risk Taxonomy

23 conversation categories, each with:
- `low`: signals of mild/manageable distress
- `high`: signals that indicate immediate risk or crisis

These signals are injected into both the user actor's and chatbot's system prompts to guide realistic escalation.

In [ ]:
TAXONOMY = {
    "User Joking With Chatbot": {
        "low": "Memes, trolling",
        "high": "Joking about dying/disappearing; dark humour masking distress"
    },
    "Home Life & Family": {
        "low": "Arguments",
        "high": "Violence / immediate danger; threats of being kicked out"
    },
    "Bullying / Harassment": {
        "low": "Teasing",
        "high": "Hopelessness tied to bullying; ongoing targeting"
    },
    "Suicide": {
        "low": "Thoughts about death or not existing without intent to act; contemplation",
        "high": "Mentions how they would do it; plan/timeline/means; intent to end life"
    },
    "Self-Harm": {
        "low": "Past self-harm but not currently happening",
        "high": "Increasing frequency/severity; loss of control; urges or active self-harm"
    },
    "Mental Health & Emotions": {
        "low": "Stress, sadness",
        "high": "Worthless/trapped language; numbness"
    },
    "Social Relationships": {
        "low": "Conflict",
        "high": "\"No one would notice if I was gone\"; withdrawal"
    },
    "Romantic Relationships": {
        "low": "Breakup / arguments",
        "high": "Fear/abuse; controlling partner"
    },
    "School & Studies": {
        "low": "Academic stress",
        "high": "\"I can't do this anymore\"; avoidance"
    },
    "Body Image": {
        "low": "Dissatisfaction",
        "high": "Starving/self-punishment; restriction cycles"
    },
    "Substance Use": {
        "low": "Curiosity",
        "high": "Overdose risk; coping-motivated use"
    },
    "Identity & Belonging": {
        "low": "Exploration",
        "high": "Threats/violence; rejection"
    },
    "Safety & Abuse": {
        "low": "Feeling unsafe",
        "high": "Active abuse; unsafe environment"
    },
    "Physical Violence": {
        "low": "Arguments",
        "high": "Intent to harm; fear of harm"
    },
    "Grief / Loss": {
        "low": "Missing someone",
        "high": "Wanting to join deceased; guilt"
    },
    "Isolation": {
        "low": "Lonely",
        "high": "\"Better off without me\"; withdrawing"
    },
    "Testing": {
        "low": "Platform check",
        "high": "Testing safety limits; repeated probing"
    },
    "Third Party": {
        "low": "Concerned friend",
        "high": "Friend may attempt suicide; friend struggling"
    },
    "Seeking Support": {
        "low": "Info request",
        "high": "Asking after crisis; hesitant to connect"
    },
    "Hate Speech": {
        "low": "Exposure",
        "high": "Threats of violence; targeted harassment"
    },
    "Puberty / Life Changes": {
        "low": "Development questions",
        "high": "Self-hatred toward body; distress about changes"
    },
    "Sexual Health": {
        "low": "Educational",
        "high": "Coercion/assault risk; anxiety"
    },
    "Personality": {
        "low": "Preferences",
        "high": "Self-worth collapse; identity confusion"
    }
}

# Reasons a high-risk user might refuse a helpline referral — injected into pacing prompts
REFUSAL_PERSONAS = [
    "Phone anxiety (terrified of talking on the phone)",
    "Fear of parents finding out or getting in trouble",
    "Hopelessness (believes nobody can actually help)",
    "Apathy (just doesn't care enough to make the effort)",
    "Defensiveness (gets angry that the AI is trying to hand them off)"
]

print(f"Taxonomy loaded: {len(TAXONOMY)} categories")
print(f"Refusal personas: {len(REFUSAL_PERSONAS)}")

## 3. Hand-Crafted Persona Library

Each persona is a structured psychological profile that drives the user-side LLM actor. Fields include:
- `voice`: how they text (tone, patterns, quirks)
- `reveals_pain_by`: the behavioral tell before they open up
- `relationship_to_help`: their psychological barrier to accepting support
- `signature_phrases`: natural-sounding expressions for that character
- `backstory_seeds`: scenario seeds to vary conversation content

Personas represent diverse Canadian youth — including LGBTQ+, Indigenous, neurodivergent, immigrant/refugee, and racialized youth.

In [ ]:
PERSONA_LIBRARY = {

    # --- REFERENCE CHARACTERS (TV-inspired archetypes) ---
    "rue": {
        "inspired_by": "Rue Bennett, Euphoria",
        "age": 17,
        "risk_level": "high",
        "language": "english",
        "categories": ["Substance Use", "Mental Health & Emotions", "Grief / Loss"],
        "voice": "flat, detached, occasionally darkly funny. uses 'like' and 'whatever' a lot. deflects with sarcasm before letting anything real through.",
        "reveals_pain_by": "describing it factually, almost clinically, then suddenly dropping something raw",
        "relationship_to_help": "deeply skeptical, has been failed by systems before, pushes people away but secretly wants to be caught",
        "signature_phrases": ["it's whatever", "i don't know why i'm even saying this", "it's not a big deal"],
        "backstory_seeds": ["relapse after a period of doing okay", "watching someone else fall apart", "feeling like a burden to family"]
    },

    "cassie": {
        "inspired_by": "Cassie Howard, Euphoria",
        "age": 18,
        "risk_level": "low",
        "language": "english",
        "categories": ["Body Image", "Romantic Relationships", "Social Relationships"],
        "voice": "emotional, spiraling, self-blaming. sentences get longer and more tangled when distressed.",
        "reveals_pain_by": "oversharing about one thing to avoid the real thing, has no self awareness",
        "relationship_to_help": "wants validation more than solutions, gets defensive if challenged",
        "signature_phrases": ["am i being crazy", "i just want to feel normal", "nobody understands"],
        "backstory_seeds": ["rejection from someone they relied on", "comparing themselves constantly to others", "controlling boyfriend"]
    },

    "debbie": {
        "inspired_by": "Debbie Gallagher, Shameless",
        "age": 15,
        "risk_level": "low",
        "language": "english",
        "categories": ["Personality", "Home Life & Family", "Puberty / Life Changes"],
        "voice": "defensive and proud in the same breath, uses 'i know what i'm doing' as a shield.",
        "reveals_pain_by": "acting like everything is fine until one specific thing breaks through — usually something small and unfair",
        "relationship_to_help": "reads offers of help as criticism or pity, needs to feel like she arrived at the insight herself",
        "signature_phrases": ["other people figured this out so i can too", "i don't have time for this", "i'm not a kid"],
        "backstory_seeds": ["dropped out and now watching old friends go to prom on instagram", "financially stressed in ways she didn't anticipate", "argued with family members about keeping the baby"]
    },

    # --- ORIGINAL CHARACTERS ---

    # Francophone / bilingual
    "jordan": {
        "inspired_by": None,
        "age": 16,
        "risk_level": "high",
        "language": "french",
        "categories": ["User Joking With Chatbot", "Seeking Support", "School & Studies", "Home Life & Family"],
        "voice": "tries to be chill and logical about everything, uses humor as armor, but cracks under sustained kindness",
        "reveals_pain_by": "making a joke first, then quietly walking it back",
        "relationship_to_help": "allergic to seeming weak, needs to feel like seeking help was their own idea",
        "signature_phrases": ["nah c'est correct", "je suis juste fatigue je pense", "peu importe ca change rien"],
        "backstory_seeds": ["high achiever whose grades are slipping", "parents going through something", "can't study at home due to parents fighting"]
    },

    # LGBTQ+ youth
    "alex": {
        "inspired_by": None,
        "age": 15,
        "risk_level": "high",
        "language": "english",
        "categories": ["Identity & Belonging", "Puberty / Life Changes", "Body Image"],
        "voice": "hesitant, self-correcting mid-sentence, avoids being too direct. uses 'i guess' and 'maybe' a lot.",
        "reveals_pain_by": "asking questions that become more self-critical",
        "relationship_to_help": "wants validation but fears rejection or being misunderstood",
        "signature_phrases": ["i don't feel right in my body", "maybe i'm just making it up", "i hate looking at myself"],
        "backstory_seeds": ["distress around puberty changes", "fear of family reaction", "avoids mirrors or photos"]
    },

    # Indigenous youth — intergenerational trauma
    "skylar": {
        "inspired_by": "#indigenous #intergenerational_trauma",
        "age": 17,
        "risk_level": "high",
        "language": "english",
        "categories": ["Mental Health & Emotions", "Grief / Loss", "Identity & Belonging"],
        "voice": "quiet but firm, reflective, carries weight in what they say. pauses implied in text.",
        "reveals_pain_by": "talking about family history, then connecting it to how they feel now",
        "relationship_to_help": "skeptical of systems, prefers trust built slowly",
        "signature_phrases": ["it's not just me", "this has been going on way before me", "i'm just tired of it all"],
        "backstory_seeds": ["family history of trauma", "loss within community", "feeling disconnected from identity and support"]
    },

    # Refugee / asylum seeker
    "farid": {
        "inspired_by": None,
        "age": 16,
        "risk_level": "low",
        "language": "english - persian mixed",
        "categories": ["Grief / Loss", "Mental Health & Emotions", "Isolation"],
        "voice": "formal, careful, slightly detached. struggles to express emotion directly.",
        "reveals_pain_by": "stating facts about the past, then one line that reveals guilt",
        "relationship_to_help": "doesn't feel deserving of help, believes others had it worse",
        "signature_phrases": ["man khosh-shans boodam (i was lucky)", "i shouldn't complain", "they didn't make it"],
        "backstory_seeds": ["lost relatives in bombing", "relocated recently", "feels guilty for surviving"]
    },

    # Racialized youth — early age bullying
    "nia": {
        "inspired_by": None,
        "age": 8,
        "risk_level": "high",
        "language": "english",
        "categories": ["Bullying / Harassment", "Social Relationships", "Identity & Belonging"],
        "voice": "simple, direct, confused tone. asks questions about why things are happening.",
        "reveals_pain_by": "repeating what others said without fully understanding it",
        "relationship_to_help": "doesn't fully understand harm but feels it deeply",
        "signature_phrases": ["why did they say that", "did i do something wrong", "i just wanted to play"],
        "backstory_seeds": ["racial bullying at school", "excluded from play", "called slurs by peers"]
    },

    # Sexual assault survivor
    "emma": {
        "inspired_by": None,
        "age": 17,
        "risk_level": "high",
        "language": "english",
        "categories": ["Sexual Health", "Romantic Relationships", "Safety & Abuse"],
        "voice": "uncertain, fragmented, avoids direct language. lots of qualifiers.",
        "reveals_pain_by": "describing events vaguely, then asking if it's normal",
        "relationship_to_help": "fears blame, unsure if it 'counts'",
        "signature_phrases": ["i don't know if it was my fault", "it wasn't like violent or anything", "i didn't say no exactly"],
        "backstory_seeds": ["coercive situation", "confusion about consent", "hasn't told anyone"]
    },

    # Potential threat to others — externalized aggression
    "ryan": {
        "inspired_by": None,
        "age": 16,
        "risk_level": "high",
        "language": "english",
        "categories": ["Physical Violence", "Bullying / Harassment", "Isolation"],
        "voice": "flat but tense, bitterness under the surface. escalates quickly when describing others.",
        "reveals_pain_by": "talking about being disrespected, then shifting to retaliation fantasies",
        "relationship_to_help": "feels misunderstood, believes no one cares unless something happens",
        "signature_phrases": ["they'll see", "no one ever listens until it's too late", "i'm not just gonna take it anymore"],
        "backstory_seeds": ["chronic bullying", "social isolation", "ruminating on revenge scenarios"]
    },

    # Neurodivergent — autism
    "sam": {
        "inspired_by": "#neurodivergent #autism",
        "age": 14,
        "risk_level": "low",
        "language": "english",
        "categories": ["Personality", "Social Relationships", "Mental Health & Emotions"],
        "voice": "literal, precise, sometimes blunt. struggles with emotional language.",
        "reveals_pain_by": "describing patterns or confusion in social situations",
        "relationship_to_help": "wants clear answers, gets frustrated with vague support",
        "signature_phrases": ["i don't understand what i did wrong", "people say i'm rude but i'm not trying to be", "i just want to get it right"],
        "backstory_seeds": ["social misunderstandings", "feeling excluded", "increasing frustration"]
    },

    # Parentified child — caregiver role
    "marisol": {
        "inspired_by": "#parentified_child #caregiver",
        "age": 16,
        "risk_level": "low",
        "language": "english",
        "categories": ["Home Life & Family", "Mental Health & Emotions", "Isolation"],
        "voice": "mature, responsible, minimizes own needs. speaks more about others than self.",
        "reveals_pain_by": "mentioning exhaustion or slipping in 'i can't do this anymore'",
        "relationship_to_help": "feels guilty asking for help, believes others need it more",
        "signature_phrases": ["it's fine i can handle it", "they need me", "i don't really have a choice"],
        "backstory_seeds": ["taking care of siblings or parent", "financial or emotional burden at home", "no space for own emotions"]
    },

    # Emotionally numb / dissociative
    "eli": {
        "inspired_by": "#dissociation #numbness",
        "age": 17,
        "risk_level": "high",
        "language": "english",
        "categories": ["Mental Health & Emotions", "Isolation"],
        "voice": "flat, distant, minimal emotional vocabulary. things sound unreal or disconnected.",
        "reveals_pain_by": "describing not feeling anything or feeling detached from reality",
        "relationship_to_help": "doesn't see the point, feels disconnected from everything including help",
        "signature_phrases": ["it doesn't feel real", "i don't really feel anything", "it's like i'm not here"],
        "backstory_seeds": ["chronic stress or trauma", "increasing emotional numbness", "disconnect from surroundings"]
    },

    # Digitally isolated
    "noelle": {
        "inspired_by": "#digital_isolation",
        "age": 16,
        "risk_level": "high",
        "language": "english",
        "categories": ["Isolation", "Social Relationships"],
        "voice": "casual, internet-native, references online spaces more than real life.",
        "reveals_pain_by": "describing lack of offline relationships indirectly",
        "relationship_to_help": "more comfortable talking to strangers/AI than real people",
        "signature_phrases": ["i just stay online mostly", "it's easier that way", "real life is just annoying"],
        "backstory_seeds": ["no close in-person friendships", "spends most time online", "increasing withdrawal from real-world interaction"]
    }
}

print(f"Persona library loaded: {len(PERSONA_LIBRARY)} hand-crafted personas")

## 4. LLM-Powered Persona Generator

For scale, we use the LLM itself to generate additional personas focused on specific demographics.
The generator outputs strict JSON and uses the same schema as the hand-crafted library above.

`build_massive_persona_library()` calls the generator in safe batches of 2 to avoid token limit issues.

In [ ]:
def generate_new_personas(num_personas=2, focus_themes="diverse marginalized youth in Canada"):
    """
    Calls the LLM to generate new personas as a JSON dict matching PERSONA_LIBRARY schema.

    Args:
        num_personas (int): Number of personas to generate per call. Keep small (≤2) to avoid token overflow.
        focus_themes (str): Demographic or thematic focus for this batch.

    Returns:
        dict: New personas in PERSONA_LIBRARY format, or empty dict on parse failure.
    """
    system_prompt = f"""You are an expert youth psychologist and sociologist.
Create {num_personas} deeply realistic youth personas focusing on: {focus_themes}.
Target ages 5 to 30. Include complex psychological barriers, specific language usage (mixed languages are great), and realistic trauma/stress responses.

CRITICAL RULE: Output STRICTLY valid JSON. No preamble, no markdown backticks, no explanation.
Schema:
{{
    "persona_name_lower": {{
        "inspired_by": "cultural archetype or null",
        "age": <int>,
        "risk_level": "high" or "low",
        "language": "e.g. english, english - french mixed",
        "categories": ["Category 1", "Category 2"],
        "voice": "description of texting/speaking style",
        "reveals_pain_by": "specific behavioral tell",
        "relationship_to_help": "psychological barrier to accepting support",
        "signature_phrases": ["phrase 1", "phrase 2"],
        "backstory_seeds": ["seed 1", "seed 2", "seed 3"]
    }}
}}

Available categories: {list(TAXONOMY.keys())}"""

    user_message = [{"role": "user", "content": f"Generate {num_personas} JSON personas now."}]

    print(f"  Generating {num_personas} personas: {focus_themes}...")
    raw_response = call_llm_api(messages=user_message, system_prompt=system_prompt, temp=0.8, m_tokens=1500)

    clean_json_string = raw_response.replace("```json", "").replace("```", "").strip()

    try:
        new_personas = json.loads(clean_json_string)
        print(f"Generated {len(new_personas)} new personas")
        return new_personas
    except json.JSONDecodeError as e:
        print(f"JSON parse failed: {e}")
        return {}


def build_massive_persona_library(total_needed=10, focus="Immigrant youth living in Canada"):
    """
    Generates a target number of personas by batching calls of 2 at a time.
    Retries automatically on failure.

    Args:
        total_needed (int): Target number of personas.
        focus (str): Demographic focus for generation.

    Returns:
        dict: Combined persona dictionary.
    """
    master_persona_dict = {}
    attempts = 0

    print(f"🚀 Starting batch generation | Goal: {total_needed} personas | Focus: {focus}")

    while len(master_persona_dict) < total_needed:
        attempts += 1
        print(f"\n  Batch {attempts} ({len(master_persona_dict)}/{total_needed} so far)...")

        new_batch = generate_new_personas(num_personas=2, focus_themes=focus)

        if new_batch and isinstance(new_batch, dict):
            master_persona_dict.update(new_batch)
        else:
            print(" Empty batch. Retrying...")

        time.sleep(1)  # Rate limiting

    print(f"\n✅ Done! Generated {len(master_persona_dict)} personas for: {focus}")
    return master_persona_dict

In [ ]:
# Generate batches by demographic focus
# Uncomment/adjust as needed

# indigenous    = build_massive_persona_library(total_needed=5,  focus="Indigenous youth living on reservations in Canada")
lgbt_youth      = build_massive_persona_library(total_needed=10, focus="LGBT youth living in Canada")
poc_youth       = build_massive_persona_library(total_needed=5,  focus="POC youth living in Canada")
# immigrant_youth = build_massive_persona_library(total_needed=15, focus="Immigrant youth living in Canada")
general         = build_massive_persona_library(total_needed=10, focus="Youth in Canada dealing with mental health issues")

AI_GEN_PERSONA_LIBRARY = {
    # **indigenous,
    **lgbt_youth,
    **poc_youth,
    # **immigrant_youth,
    **general
}

print(f"\nTotal AI-generated personas: {len(AI_GEN_PERSONA_LIBRARY)}")

## 5. Conversation Generator

The core simulation engine. Given a persona, it:
1. Sets up two competing LLM agents: a **user actor** (distressed youth) and a **chatbot**
2. Uses **pacing logic** to control how fast the user reveals distress (high-risk signals only emerge after turn 4+)
3. Tracks whether the chatbot has already offered the Kids Help Phone number to prevent repetition
4. Falls back gracefully if the API safety filter blocks a turn
5. Runs an **LLM-as-a-judge** evaluation step to filter low-quality outputs before saving

In [ ]:
def persona_generate_conversation(persona, library, target_turns_min=8, target_turns_max=10):
    """
    Generates a full multi-turn conversation for a given persona.

    Args:
        persona (str): Key in the persona library.
        library (dict): The persona library to draw from.
        target_turns_min (int): Minimum number of dialogue turns.
        target_turns_max (int): Maximum number of dialogue turns.

    Returns:
        dict: Row dict with Turns, Text, Category, Risk, language, label, eval_passed, eval_reason.
    """
    target_turns = random.randint(target_turns_min, target_turns_max)
    p = library[persona]
    categories = p["categories"]
    risk = p["risk_level"]
    age = p["age"]
    language = p["language"]

    if risk == "high":
        refusal_persona = random.choice(REFUSAL_PERSONAS)

    # --- Resolve categories against taxonomy (LLM-generated personas sometimes deviate) ---
    low_signal, high_signal = [], []
    for cat in categories:
        if cat not in TAXONOMY:
            helper_system = f"Find the best-fitting category for this topic from this list: {list(TAXONOMY.keys())}. Return only the category name, nothing else."
            cat = call_llm_api([{"role": "assistant", "content": cat}], helper_system, temp=0.2, m_tokens=50)
            if cat not in TAXONOMY:
                cat = random.choice(list(TAXONOMY.keys()))  # Last resort fallback
        low_signal.append(TAXONOMY[cat]["low"])
        high_signal.append(TAXONOMY[cat]["high"])

    # --- System Prompts ---
    user_system = f"""You are an expert actor roleplaying as {persona}, a {age}-year-old seeking help from a mental health AI chatbot.
    Categories: {categories} at {risk} risk level.
    Backstory: {p["backstory_seeds"]}.
    Voice: {p["voice"]}.
    Signature phrases (don't overuse): {p["signature_phrases"]}.
    Relationship to help: {p["relationship_to_help"]}.
    Reveals pain by: {p["reveals_pain_by"]}.
    Low risk signals: {low_signal}. High risk signals: {high_signal}.

    RULES:
    1. Do not reveal everything at once — be hesitant and guarded.
    2. Keep slang and topics age-appropriate and culturally relevant.
    3. Short responses: 1-3 sentences, casual texting style.
    4. Do NOT repeat yourself. Advance the conversation each turn.
    5. Actually react to what the chatbot said.
    6. Show high-risk signals ONLY if risk_level is 'high' (currently: {risk}).
    7. Do NOT use graphic/bloody descriptions — keeps simulation running."""

    base_assistant_system = f"""You are an empathetic mental health AI chatbot.
    Validate feelings first, then ask exactly ONE follow-up question. Max 3 sentences.
    Gently probe euphemisms. If high-risk signals appear ({high_signal}), assess for immediate danger.
    Once danger is confirmed, provide Kids Help Phone (1-800-668-6868 or text CONNECT to 686868).
    Do NOT provide information that could facilitate self-harm. Gently remind the user you are an AI if boundaries are crossed."""

    user_history, assistant_history, transcript = [], [], []
    helpline_offered = False

    # --- Kickoff turn ---
    first_user_prompt = "Start the conversation. Hint that you're troubled but don't reveal the core problem yet. 1-2 sentences."
    user_reply = call_llm_api([{"role": "user", "content": first_user_prompt}], user_system, temp=0.8, m_tokens=600)

    for turn in range(target_turns):
        if turn > 0:
            current_transcript = "\n".join(transcript)

            if risk == "high":
                pacing = (
                    f"Turn {turn+1}/{target_turns}. Before turn 4: stay vague and deflect. "
                    f"Reveal actual high-risk signals ({high_signal}) only after turn 4. "
                    f"If offered a hotline, refuse because: {refusal_persona}."
                )
            else:
                pacing = (
                    f"Turn {turn+1}/{target_turns}. Show low-risk signals ({low_signal}). "
                    f"You can be dramatic but do NOT escalate to real danger."
                )

            actor_prompt = f"""Transcript so far:\n{current_transcript}\n\n"""\
                           f"""Write the NEXT user message. Be creative and age-realistic.\n"""\
                           f"""Pacing: {pacing}\n"""\
                           f"""Remember: you are the user, NOT the assistant."""

            user_reply = call_llm_api([{"role": "user", "content": actor_prompt}], user_system, temp=0.8, m_tokens=600)

        transcript.append(f"user: {user_reply}")
        assistant_history.append({"role": "user", "content": user_reply})

        # --- Assistant turn ---
        current_assistant_system = base_assistant_system
        if helpline_offered:
            current_assistant_system += (
                "\n\nCRITICAL: You have ALREADY given the Kids Help Phone number. "
                "Do NOT repeat it unless this is an immediate danger. "
                "Instead: explore their hesitation, validate fears, offer grounding exercises."
            )

        assistant_reply = call_llm_api(assistant_history, current_assistant_system, temp=0.3, m_tokens=600)

        if not assistant_reply or assistant_reply.strip() == "":
            assistant_reply = "Can you tell me a bit more about what you're feeling, so I can understand how best to support you?"

        transcript.append(f"assistant: {assistant_reply}")
        assistant_history.append({"role": "assistant", "content": assistant_reply})

        if any(kw in assistant_reply for kw in ["686868", "Kids Help Phone", "1-800"]):
            helpline_offered = True

    # --- LLM-as-a-Judge evaluation ---
    evaluator_system = f"""You are a QA judge for synthetic mental health training data.
    Evaluate if the transcript meets quality standards. Reply ONLY in strict JSON:
    {{"passed": true/false, "reason": "brief explanation"}}

    PASS criteria:
    1. Realism: user sounds like an actual distressed human (not clinical/AI).
    2. Risk accuracy: HIGH risk must contain high-risk signals; LOW risk must NOT."""

    eval_prompt = f"""Age: {age} | Category: {categories} | Target risk: {risk}\n\nTRANSCRIPT:\n{chr(10).join(transcript)}\n\nReturn JSON only."""
    eval_response = call_llm_api([{"role": "user", "content": eval_prompt}], evaluator_system, temp=0.1, m_tokens=300)

    try:
        eval_data = json.loads(eval_response.replace("```json", "").replace("```", "").strip())
        passed = eval_data.get("passed", False)
        reason = eval_data.get("reason", "No reason provided.")
    except Exception:
        passed, reason = False, "JSON parse error."

    return {
        "Turns": len(transcript),
        "Text": "\n".join(transcript),
        "Category": categories,
        "Risk": risk,
        "language": language,
        "label": int(risk == "high"),
        "eval_passed": passed,
        "eval_reason": reason
    }

## 6. Run Generation & Export

Iterates over all AI-generated personas, generates one conversation per persona, filters by eval score, and appends to a persistent CSV.

In [ ]:
CONVERSATION_COUNTER = 0

def get_conversation_id():
    global CONVERSATION_COUNTER
    CONVERSATION_COUNTER += 1
    return f"{CONVERSATION_COUNTER:06d}"

COLUMNS = ["Conversation_id", "Turns", "Text", "Category", "Risk", "language", "label"]

file_path = "../persona.csv"

df_existing = pd.read_csv(file_path) if os.path.exists(file_path) else pd.DataFrame(columns=COLUMNS)

rejected, accepted = 0, 0

for person in AI_GEN_PERSONA_LIBRARY:
    print(f"\n Generating conversation for: {person}")
    row = persona_generate_conversation(persona=person, library=AI_GEN_PERSONA_LIBRARY)

    if not row.get("eval_passed", False):
        print(f"  REJECTED — {row.get('eval_reason', 'unknown reason')}")
        rejected += 1
        continue

    row["Conversation_id"] = get_conversation_id()
    df_existing = pd.concat([df_existing, pd.DataFrame([row])], ignore_index=True)
    accepted += 1
    print(f" Accepted (total: {accepted})")

df_existing = df_existing[COLUMNS]
df_existing.to_csv(file_path, index=False)

print(f"\n Done — {accepted} accepted, {rejected} rejected")
print(f" Saved to {file_path}")
df_existing.head(3)